In [ ]:
!pip install --upgrade huggingface_hub

from huggingface_hub import login
login()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

In [ ]:
!pip install transformers datasets evaluate rouge_score

In [ ]:
from datasets import load_dataset

# Load the California portion of the BillSum dataset
dataset = load_dataset("billsum", split="ca_test")

# Look at the first legal text and its summary
print(f"Legal Text: {dataset[0]['text'][:200]}...")
print(f"Official Summary: {dataset[0]['summary']}")

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Define the model name
model_name = "facebook/bart-large-cnn"

# Load the tokenizer and model, ensuring the model is moved to the specified device
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# 1. Access the specific string (the first legal document)
single_document = dataset[0]['text']

# 2. Tokenize the single string
inputs = tokenizer(
    single_document,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(device) # Ensure inputs are on the device

# 3. Generate the summary
summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=4,
    min_length=50,
    max_length=150,
    early_stopping=True
)

# 4. Decode the result back to English
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("AI Generated Legal Summary")
print(summary_text)